<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c36-header.png" alt="Nextia Learning · Text Classification and Routing: Send Each Message to the Right Place" width="100%">

# Zero-shot classification with an LLM

**[Zero-shot classification with an LLM](https://learning.nextia-ai.com/courses/text-routing/m05/zero-shot-classification-with-an-llm/)** · Text Classification and Routing: Send Each Message to the Right Place · Module 5, lesson 1 of 5 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** classify messages with a hosted LLM from the route names alone, with a JSON answer that you check, and to compare its accuracy, its cost and its time with the local models. The LLM answers were recorded once, so you need no key.

| | |
|---|---|
| **Time** | About 35 minutes |
| **Needs** | An internet connection for the setup cells. No account and no key: the notebook replays recorded answers. |
| **You should know** | Structured output, from the LLM API course (helpful, not required). The comparison table, from [Meet the inbox](https://learning.nextia-ai.com/courses/text-routing/m01/meet-the-inbox/). |
| **Tested** | Python 3.12 with pandas 3.1.1, scikit-learn 1.9.1 and matplotlib 3.11.2, in Jupyter on a laptop CPU (Apple M4, 16 GB), from a fresh start (*Restart and run all*) |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m05/zero-shot-classification-with-an-llm/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C36/M05-L01-zero-shot-classification-with-an-llm/zero-shot-classification-with-an-llm-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, scikit-learn 1.9.1, and matplotlib for the charts. It needs an internet connection and takes about a minute the first time.

You should see one line per package, with the versions above. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. Restarting is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 scikit-learn==1.9.1 matplotlib==3.11.2
import warnings; warnings.filterwarnings("ignore")
import pandas; print("pandas", pandas.__version__)
import sklearn; print("scikit-learn", sklearn.__version__)

## Setup: get the practice data

Run the next cell. It downloads the Larkfield inbox of this course (about 9 MB) into a folder `larkfield-inbox/data`, and the course's helper `routing.py`, and checks each file's checksum. It needs an internet connection and no account, and takes about 10 seconds.

You should see `Ready: 10000 messages (6000 train, 2000 validation, 2000 test), all synthetic.` If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `larkfield-inbox` (in Colab: the **Files** panel on the left), then run the cell again.

The Larkfield inbox is synthetic: we made it for this course. It has no real people. The answers of the hosted models were recorded once, so you need no key.

In [ ]:
# Setup: download the course data into larkfield-inbox/ and check it.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C36/"
FILES = {
    "data/corrections.csv": "be3088aecd9b7ba5d07ea222b79e06a476806eba81bc80c2f33da695878e5cc0",
    "data/costs.json": "9a2d566f062c3382b8884ef07756f886179c69a3525d80e3d4c15c0c4fd4507e",
    "data/dataset-card.md": "fce0d2e602002a2ea2c6fc26c1d2ab9c3d1a30906f40fa4a426180c3639e99af",
    "data/double_labelled.csv": "ba4a8dadc3c1ba8d48af893124bd33fb7a895f917a2add211486e3dbd353dfc9",
    "data/guidelines.md": "8ab83de04998b9017b1aabce00be04084e989af75aff5a9abdd760f4d3cb24b2",
    "data/launch_labels_sample.csv": "600c689325b7c14e56f4a27e92e395eb0b2e965411b1abf9d724720c8b485029",
    "data/launch_month.csv": "7ef3cc2c5fa910023c6330c1e895744358846ce451e551b86be8c9abe1c4289f",
    "data/messages.csv": "266ca23c0551249fc114fc74253103ea8602a04be1fe31d1b7f91bc94ce6dd5e",
    "data/old_queue.csv": "e3cde5d19d3ebe2e6e0e067693c76796c0abbf89df0dc93dc310ad3fe42a5cae",
    "data/predictions/embeddings-newer.csv": "3c7ffe5b5371ca60c3b6bedc17b307142bcfd77f06c97db7ed267b703a50a8d2",
    "data/predictions/embeddings-newer.json": "dcd537f2971b9a48b5354cc9fbb48b87689b83b3a479da9acd119701c2fc19f6",
    "data/predictions/embeddings-older.csv": "a06c490ac17cbc1c3c75380f6cdf7775483de92961cdbb78b2a7914256926539",
    "data/predictions/embeddings-older.json": "db658bbd13861529122807726b6b41e1da9c70e3b55aba55cce9e868a795e8b4",
    "data/predictions/encoder-finetuned.csv": "abced2276671bfd413319573907a48c60952310e50978dba96c0dde0332b7d39",
    "data/predictions/encoder-finetuned.json": "1f7c89271a199c1d659bdc29d8ac0592558a1424218bd0519bbbde9840212b94",
    "data/predictions/gliclass-zero-shot.csv": "2c22e5036de94a376f7c0ccf22603c4d7c624135c8a4965123c9492748814008",
    "data/predictions/gliclass-zero-shot.json": "95ad2acafd48ad2c24516333dba7e78a3eabc7ca616c4053ce2f144f56cbaaec",
    "data/predictions/llm-few-shot.csv": "bcdaee3550f78ebb672465314bbcc9ad26283aaea69265a979739cf6a5f965e2",
    "data/predictions/llm-few-shot.json": "104b446894a0450a7e02fcd07c6db44f3e1a5486002b388374ce6dae4a744a12",
    "data/predictions/llm-zero-shot.csv": "5518e2b668a110b572687313bc13bd689774dff96c0a1614e1cd92e14ddf9ce7",
    "data/predictions/llm-zero-shot.json": "9a72ce600d0d858b00dd3b37247a305f9e8b3fc2fef3b8dbe60c0871b0c1b355",
    "data/predictions/local-llm-prompted.csv": "5bb0d84f8ee9b5086818d3525e1da3f8172b38b7875b10d1d3021b3ad5889f33",
    "data/predictions/local-llm-prompted.json": "ae328379d5ea294b3b59845ee143f24776787e748bc14c165b5e8e9ed4ff7158",
    "data/predictions/local-llm-scores.csv": "f85f570075ea9288fd14a6c3955d4310ba740814f3de472e8a263cd4f9682a5f",
    "data/predictions/local-llm-scores.json": "67a7e07ce065934a947f59c1c36d416eec76f8827f01500ce82ba6e6c4d9b99e",
    "data/predictions/nli-zero-shot.csv": "ecafb001370e5cf2904ce685683beafc86a2c290cd1da80438abb9912d099d3c",
    "data/predictions/nli-zero-shot.json": "e271f3b12ad86c6308a74c0b5f8b6bf80c6534cf18c41af5c86c29b184ab53cd",
    "data/prices.json": "9e98f9056d9d1e53d76a039da9b10d2b4fa54c8a5b35eb6cae32a2abdee7af38",
    "data/recordings/README.md": "d99eb50666045cc2ccf596e100103244fe1513b488fce46e26462a2666053ffc",
    "data/recordings/llm-few-shot.jsonl": "259a93f1d6dcbcb167cd45796c8e09a914e56c7741f14bd0aee53c996e8349d4",
    "data/recordings/llm-labels.jsonl": "cd6c670a23ee78cd3d8b51e5907ebae9d86448aa6236767c11d55227616fa8cc",
    "data/recordings/llm-reasoning.jsonl": "fc8ae631121f3240ce378df429af767ac52ea6e1560aa409de09ffc99b092300",
    "data/recordings/llm-zero-shot.jsonl": "957edfb8811e177c83bf23772f0da561659cf50728ffc95597f85cd7680aed86",
    "data/rules.json": "6f95d5e7b87869ec7975b198a3777cdd96a0db6a869554d922889479c87b2066",
    "routing.py": "8de6562d53415e5c3d8e74f44df9e8838cb71085011b29244187bde5a6f4d93c",
}
HERE = Path.cwd()
# The notebook works in its own folder, marked with a file, so it never
# writes into your own project if you open the notebook there.
if Path(".nextia-notebook").exists():  # the cell ran before: start from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
PROJECT = Path("larkfield-inbox").resolve()
(PROJECT / "data").mkdir(parents=True, exist_ok=True)
(PROJECT / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

for name, expected in FILES.items():
    path = PROJECT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists() and sha256(path) == expected:
        continue
    local = HERE / name  # a copy next to the notebook, if there is one
    if local.exists():
        shutil.copy(local, path)
    else:
        download(LABS + name, path)
    if sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder larkfield-inbox and run again.")

os.chdir(PROJECT)  # work in the project folder, as in the lesson
import csv
with open("data/messages.csv", encoding="utf-8") as f:
    splits = [row["split"] for row in csv.DictReader(f)]
print(f"Ready: {len(splits)} messages ({splits.count('train')} train, {splits.count('validation')} validation, "
      f"{splits.count('test')} test), all synthetic.")

## Setup: helpers

The next cell loads pandas as `pd`, numpy as `np` and the course helper as `rt`, reads the three parts of the inbox (`train`, `validation`, `test`), and defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

Run the cell. It prints nothing.

In [ ]:
# Helpers: the data, the course helper, and checks that do not show the answer.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import routing as rt
warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 160)

messages = pd.read_csv("data/messages.csv")
train = messages[messages["split"] == "train"].reset_index(drop=True)
validation = messages[messages["split"] == "validation"].reset_index(drop=True)
test = messages[messages["split"] == "test"].reset_index(drop=True)

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple)):
            return [norm(x) for x in v]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
        return
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. The prompt and the answer schema

The course recorded one answer for each validation and test message from a small hosted model (the Azure deployment `chat-small`, model `gpt-6-luna`). Each call sent this **system prompt** and the message:

> You route customer messages for Larkfield, an online shop for home and garden products. Choose exactly one route for the message: orders, delivery, returns_refunds, billing, product, account, needs_person. Messages can be in English or French. Answer in JSON with the route, your confidence from 0 to 1, and a reason of at most 12 words.

The model had to answer with JSON that matches a **schema**: `route` (one of the seven route names), `confidence` (a number) and `reason` (a short text). This is **structured output**: the provider checks the shape of the answer.

## 2. Check every answer

A schema checks the shape. Your code must still check the values: a known route, and a confidence from 0 to 1. A bad answer goes to a person.

> **Predict.** Which of these four answers pass the check?

In [ ]:
def check(answer):
    """A valid answer: JSON, a known route, a confidence from 0 to 1. Anything else goes to review."""
    try:
        a = json.loads(answer)
        route, confidence = a["route"], float(a["confidence"])
    except (ValueError, KeyError, TypeError):
        return "review", 0.0
    if route not in rt.ROUTES or not 0 <= confidence <= 1:
        return "review", 0.0
    return route, confidence

for answer in ['{"route": "billing", "confidence": 0.9, "reason": "double charge"}',
               'Billing, I think.',
               '{"route": "refunds", "confidence": 0.8, "reason": "refund"}',
               '{"route": "delivery", "confidence": 95, "reason": "late"}']:
    print(check(answer), "<-", answer)

> **Check.** Only the first answer passes. "refunds" is not a route name (the route is `returns_refunds`), and 95 is not a confidence from 0 to 1.

## 3. The recorded answers

The next cell reads the recorded answers for the test messages, checks each one again, and scores them with the course's columns. The run cost is the mean cost of a call (its tokens at the price of the recording) × 1,000; the time is the median time of a call.

In [ ]:
answers = pd.read_json("data/recordings/llm-zero-shot.jsonl", lines=True).set_index("message_id")
llm = answers.loc[test["message_id"]]
checked = [check(a) for a in llm["answer"]]
llm_route = [r for r, _ in checked]
print(f"valid answers: {np.mean([r != 'review' for r in llm_route]):.1%}; model: {llm['model'].iloc[0]}")
rows = [rt.score("hosted LLM, zero-shot (recorded)", test, llm_route,
                 run_per_1000=llm["usd"].mean() * 1000, ms_per_message=llm["latency_s"].median() * 1000)]
emb, meta = rt.predictions("embeddings-older", test)  # a measured local method, for comparison
rows.append(rt.score("embeddings MiniLM + logistic regression", test, emb["prediction"],
                     run_per_1000=meta["run_per_1000"], ms_per_message=meta["ms_per_message"]))
rt.show(rows)

> **Check.** You should see 100.0% valid answers, and the LLM with accuracy 0.765 (English 0.757, French 0.784) and a total of 1049.0 dollars per 1,000 messages.

The LLM saw no Larkfield label and no example. It knows only the route names. Where does it go wrong? The next cell lists its most common mistakes.

In [ ]:
pd.crosstab(test["route"], pd.Series(llm_route, index=test.index), rownames=["label"], colnames=["LLM"])

Read the column `needs_person`: the model sends almost 300 messages there that a team should answer. With the route names alone, it reads every upset customer as "a person must read it first". It also sends most account takeovers elsewhere (see the `takeover` column). People learn where the line is from the guidelines; the next lesson gives the guidelines and a few examples to the LLM.

The comparison table says it plainly: here the LLM with the route names alone is **less accurate** than the frozen embeddings with logistic regression, and it costs more per 1,000 messages. "An LLM is always the most accurate classifier" is not true here.

## 4. Your turn: the cost of 1,000 messages from the tokens

`data/prices.json` has the price of the model, in dollars per 1 million tokens.

> **Your turn.** Compute the cost of 1,000 test messages from the **mean** input and output tokens of the recorded answers (`llm["tokens_in"]`, `llm["tokens_out"]`) and the `chat-small` prices. Put it in `cost_1000`, rounded to 4 decimals.

In [ ]:
cost_1000 = ...  # dollars for 1,000 messages

In [ ]:
expect('cost_1000', cost_1000, 0.0351, tolerance=0.001)

At these prices, the model's cost is small next to the cost of its mistakes. The price is not the only cost: every message also leaves Larkfield, waits for the network, and counts against the provider's rate limit. Module 5, lesson 5 compares all of them.

## 5. Failure case: the confidence in the answer

The model also wrote a confidence. Is a written "0.95" right 95 times in 100?

In [ ]:
conf = pd.Series([c for _, c in checked], index=test.index)
right = pd.Series(llm_route, index=test.index) == test["route"]
summary = pd.DataFrame({"messages": conf.round(2).value_counts(), "accuracy": right.groupby(conf.round(2)).mean().round(3)})
summary.sort_values("messages", ascending=False).head(6)

> **Check.** The most common written confidence is 0.99, with an accuracy of 86.6% at that value.

A number that the model **writes** is part of its text. It is not a probability that anyone measured. Module 6 shows how to get a real probability from a model, and Module 8 how to check one.

## Optional: run it live with your own key

This cell is off by default. It sends **one** synthetic message to a model of your own (OpenAI or any provider with the same API), with the same prompt. It costs a fraction of a cent. Set `LIVE = True` and your key in the environment variable `OPENAI_API_KEY` first.

In [ ]:
LIVE = False
if LIVE:
    %pip install -q openai
    from openai import OpenAI
    client = OpenAI()
    r = client.chat.completions.create(model="gpt-4.1-mini", messages=[
        {"role": "system", "content": 'You route customer messages for Larkfield, an online shop for home and garden products. Choose exactly one route for the message: orders, delivery, returns_refunds, billing, product, account, needs_person. Messages can be in English or French. Answer in JSON with the route, your confidence from 0 to 1, and a reason of at most 12 words.'},
        {"role": "user", "content": "Message:\n" + test["text"].iloc[0]}],
        response_format={"type": "json_object"})
    print(check(r.choices[0].message.content), "label:", test["route"].iloc[0])

## Next

The [lesson page](https://learning.nextia-ai.com/courses/text-routing/m05/zero-shot-classification-with-an-llm/) has the diagrams and the knowledge checks that count toward your certificate.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Few-shot examples and label descriptions](https://learning.nextia-ai.com/courses/text-routing/m05/few-shot-examples-and-label-descriptions/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/text-routing/m05/zero-shot-classification-with-an-llm/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.